In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject, transform_bounds
from rasterio.transform import from_bounds
from rasterio.crs import CRS
from rasterio.features import geometry_mask
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import warnings

warnings.filterwarnings("ignore")


In [ ]:
BASE_DIR = Path("D:\\Personal\\SPARRSO Fellowship\\GIS")
LAYERS_DIR = str(BASE_DIR / "V2/FINAL LAYERS")
AHP_CSV = str(BASE_DIR / "AHP/AHP.csv")
AOI_SHP = str(BASE_DIR / "DBA.shp")
OUTPUT_DIR = str(BASE_DIR / "V2/OUTPUT")
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_SUITABILITY = os.path.join(OUTPUT_DIR, "Solar_Irrigation_Suitability.tif")
OUTPUT_CLASSIFIED = os.path.join(
    OUTPUT_DIR, "Solar_Irrigation_Suitability_Classified.tif"
)
TARGET_RES = 30
TARGET_CRS = CRS.from_epsg(32645)
NODATA = -9999.0
NODATA_THR = -1e30


In [ ]:
LAYER_FILES = {
    "DNI": "DNI.tif",
    "LULC": "LULC.tif",
    "R_Rabi": "Rabi_PR.tif",
    "R_Kharif1": "K1_PR.tif",
    "R_Kharif2": "K2_PR.tif",
    "Temperature": "TEMPERATURE.tif",
    "DR": "DIS_F_ROAD.tif",
    "DW": "DIS_F_RIVERS.tif",
    "Relative_Relief": "RELATIVE_RELIEF.tif",
    "Cloudy_Days": "CLOUDY_DAYS.tif",
}
missing = []
for key, fname in LAYER_FILES.items():
    fpath = os.path.join(LAYERS_DIR, fname)
    if not os.path.exists(fpath):
        missing.append(key)
if missing:
    raise FileNotFoundError(f"Missing input layers: {missing}")


In [ ]:
ahp_df = pd.read_csv(AHP_CSV, header=None, names=["Factor", "Weight"])
ahp_df["Factor"] = ahp_df["Factor"].str.strip()
ahp_df["Weight"] = pd.to_numeric(ahp_df["Weight"], errors="coerce")
WEIGHTS = dict(zip(ahp_df["Factor"], ahp_df["Weight"].astype(float)))
total_weight = sum(WEIGHTS.values())
WEIGHTS = {k: v / total_weight for k, v in WEIGHTS.items()}
print(pd.Series(WEIGHTS, name="Weight").to_string())


In [ ]:
aoi = gpd.read_file(AOI_SHP).to_crs(epsg=32645)
aoi_bounds = aoi.total_bounds
ref_left = aoi_bounds[0]
ref_bottom = aoi_bounds[1]
ref_right = aoi_bounds[2]
ref_top = aoi_bounds[3]
new_width = int(np.ceil((ref_right - ref_left) / TARGET_RES))
new_height = int(np.ceil((ref_top - ref_bottom) / TARGET_RES))
new_transform = from_bounds(
    ref_left, ref_bottom, ref_right, ref_top, new_width, new_height
)
REF_PROFILE = {
    "driver": "GTiff",
    "dtype": "float32",
    "width": new_width,
    "height": new_height,
    "count": 1,
    "crs": TARGET_CRS,
    "transform": new_transform,
    "nodata": NODATA,
    "compress": "lzw",
}
REF_SHAPE = (new_height, new_width)
aoi_geometries = aoi.geometry.values
AOI_MASK = ~geometry_mask(
    aoi_geometries, transform=new_transform, invert=False, out_shape=REF_SHAPE
)


In [ ]:
def build_mask(arr, file_nodata=None):
    mask = np.isnan(arr) | (arr < NODATA_THR) | (arr == NODATA)
    if file_nodata is not None:
        mask |= arr == file_nodata
    return mask


def read_and_resample(filepath):
    with rasterio.open(filepath) as src:
        file_nodata = src.nodata
        raw = src.read(1).astype(np.float32)
        raw[np.isnan(raw)] = NODATA
        raw[raw < NODATA_THR] = NODATA
        if file_nodata is not None:
            raw[raw == file_nodata] = NODATA
        data = np.full(REF_SHAPE, NODATA, dtype=np.float32)
        reproject(
            source=raw,
            destination=data,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=REF_PROFILE["transform"],
            dst_crs=TARGET_CRS,
            resampling=Resampling.bilinear,
            src_nodata=NODATA,
            dst_nodata=NODATA,
        )
    return (data, file_nodata)


def reclass_quantile(arr, reverse=False, file_nodata=None):
    mask = build_mask(arr, file_nodata) | ~AOI_MASK
    valid = arr[~mask]
    result = np.full(arr.shape, NODATA, dtype=np.float32)
    if len(valid) == 0:
        return result
    qs = np.percentile(valid, [0, 20, 40, 60, 80, 100])
    qs = np.unique(qs)
    if len(qs) < 2:
        result[~mask] = 3
        return result
    classes = np.digitize(arr, qs[1:-1], right=True) + 1
    classes = np.clip(classes, 1, 5).astype(np.float32)
    if reverse:
        classes = 6 - classes
    result[~mask] = classes[~mask]
    return result


def reclass_lulc(arr, score_map, file_nodata=None):
    mask = build_mask(arr, file_nodata) | ~AOI_MASK
    result = np.full(arr.shape, NODATA, dtype=np.float32)
    for code, score in score_map.items():
        result[np.round(arr).astype(int) == code] = score
    result[mask] = NODATA
    return result


In [ ]:
LULC_SCORE_MAP = {1: 2, 2: 4, 3: 3, 4: 5, 5: 1}


In [ ]:
RECLASSIFIED = {}
for key, fname in LAYER_FILES.items():
    fpath = os.path.join(LAYERS_DIR, fname)
    arr, fnd = read_and_resample(fpath)
    if key == "DNI":
        rc = reclass_quantile(arr, reverse=False, file_nodata=fnd)
    elif key == "LULC":
        rc = reclass_lulc(arr, LULC_SCORE_MAP, file_nodata=fnd)
    elif key in ("R_Rabi", "R_Kharif1", "R_Kharif2"):
        rc = reclass_quantile(arr, reverse=True, file_nodata=fnd)
    elif key == "Temperature":
        rc = reclass_quantile(arr, reverse=False, file_nodata=fnd)
    elif key == "DR":
        rc = reclass_quantile(arr, reverse=True, file_nodata=fnd)
    elif key == "DW":
        rc = reclass_quantile(arr, reverse=False, file_nodata=fnd)
    elif key == "Relative_Relief":
        rc = reclass_quantile(arr, reverse=False, file_nodata=fnd)
    elif key == "Cloudy_Days":
        rc = reclass_quantile(arr, reverse=True, file_nodata=fnd)
    else:
        rc = reclass_quantile(arr, reverse=False, file_nodata=fnd)
    RECLASSIFIED[key] = rc
    valid = rc[rc != NODATA]
    unique_c, counts_c = np.unique(valid.astype(int), return_counts=True)
    dist = {int(u): int(c) for u, c in zip(unique_c, counts_c)}


In [ ]:
CLASS_CMAP = mcolors.ListedColormap(
    ["#d73027", "#fc8d59", "#ffffbf", "#91cf60", "#1a9850"]
)
CLASS_NORM = mcolors.BoundaryNorm([0.5, 1.5, 2.5, 3.5, 4.5, 5.5], CLASS_CMAP.N)
CLASS_LABELS = [
    "1 - Most Unsuitable",
    "2 - Mod. Unsuitable",
    "3 - Neutral",
    "4 - Mod. Suitable",
    "5 - Most Suitable",
]
CLASS_COLORS = ["#d73027", "#fc8d59", "#ffffbf", "#91cf60", "#1a9850"]
ncols = 3
nrows = int(np.ceil(len(RECLASSIFIED) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 5))
axes = axes.flatten()
for idx, (key, arr) in enumerate(RECLASSIFIED.items()):
    ax = axes[idx]
    plot_arr = np.where(arr == NODATA, np.nan, arr)
    ax.imshow(plot_arr, cmap=CLASS_CMAP, norm=CLASS_NORM, interpolation="nearest")
    ax.set_title(f"{key}\n(Weight: {WEIGHTS.get(key, 0) * 100:.2f}%)", fontsize=11)
    ax.axis("off")
for j in range(idx + 1, len(axes)):
    axes[j].set_visible(False)
legend_patches = [
    Patch(facecolor=c, label=l) for c, l in zip(CLASS_COLORS, CLASS_LABELS)
]
fig.legend(
    handles=legend_patches,
    loc="lower center",
    ncol=5,
    fontsize=10,
    frameon=True,
    bbox_to_anchor=(0.5, -0.02),
)
fig.suptitle(
    "Reclassified Suitability Layers\n(Solar Irrigation Analysis)",
    fontsize=14,
    fontweight="bold",
    y=1.01,
)
plt.tight_layout()
plt.savefig(
    os.path.join(OUTPUT_DIR, "Reclassified_Layers.png"), dpi=150, bbox_inches="tight"
)
plt.show()


In [ ]:
nodata_mask = ~AOI_MASK.copy()
for arr in RECLASSIFIED.values():
    nodata_mask |= arr == NODATA
weighted_sum = np.zeros(REF_SHAPE, dtype=np.float32)
for key, arr in RECLASSIFIED.items():
    weight = WEIGHTS.get(key, 0)
    if weight == 0:
        continue
    safe_arr = np.where(arr == NODATA, 0.0, arr)
    weighted_sum += safe_arr * weight
weighted_sum[nodata_mask] = NODATA
valid = weighted_sum[~nodata_mask]


In [ ]:
final_classified = np.full(REF_SHAPE, NODATA, dtype=np.float32)
valid_mask = ~nodata_mask
scores = weighted_sum[valid_mask]
vmin, vmax = (scores.min(), scores.max())
interval = (vmax - vmin) / 5
breaks = [vmin + interval * i for i in range(1, 5)]
classified = np.digitize(scores, breaks, right=False) + 1
classified = np.clip(classified, 1, 5)
final_classified[valid_mask] = classified
unique, counts = np.unique(classified, return_counts=True)
total_valid = int(valid_mask.sum())
suit_desc = {
    1: "Most Unsuitable",
    2: "Moderately Unsuitable",
    3: "Neutral",
    4: "Moderately Suitable",
    5: "Most Suitable",
}


In [ ]:
def save_raster(arr, profile, filepath):
    prof = profile.copy()
    prof.update(dtype=arr.dtype, count=1)
    with rasterio.open(filepath, "w", **prof) as dst:
        dst.write(arr, 1)


save_raster(weighted_sum.astype(np.float32), REF_PROFILE, OUTPUT_SUITABILITY)
classified_profile = REF_PROFILE.copy()
classified_profile.update(dtype="int16", nodata=-9999)
save_raster(final_classified.astype(np.int16), classified_profile, OUTPUT_CLASSIFIED)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
ax = axes[0]
plot_ws = np.where(weighted_sum == NODATA, np.nan, weighted_sum)
im = ax.imshow(plot_ws, cmap="RdYlGn", interpolation="nearest")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Weighted Suitability Score")
ax.set_title("Weighted Suitability Score\n(Continuous)", fontsize=13, fontweight="bold")
ax.axis("off")
ax = axes[1]
plot_cl = np.where(final_classified == NODATA, np.nan, final_classified)
ax.imshow(plot_cl, cmap=CLASS_CMAP, norm=CLASS_NORM, interpolation="nearest")
ax.set_title(
    "Solar Irrigation Suitability Map\n(5-Class)", fontsize=13, fontweight="bold"
)
ax.axis("off")
legend_patches = [
    Patch(facecolor=c, label=l) for c, l in zip(CLASS_COLORS, CLASS_LABELS)
]
axes[1].legend(
    handles=legend_patches,
    loc="lower left",
    fontsize=9,
    framealpha=0.9,
    title="Suitability Class",
    title_fontsize=10,
)
fig.suptitle(
    "Solar Irrigation Suitability Analysis\n(AHP Weighted Overlay : 30 m Resolution)",
    fontsize=15,
    fontweight="bold",
)
plt.tight_layout()
plt.savefig(
    os.path.join(OUTPUT_DIR, "Solar_Suitability_Map.png"), dpi=200, bbox_inches="tight"
)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
keys = list(WEIGHTS.keys())
vals = [WEIGHTS[k] * 100 for k in keys]
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(keys)))
bars = ax.barh(keys, vals, color=colors, edgecolor="grey", linewidth=0.5)
ax.bar_label(bars, fmt="%.2f%%", padding=3, fontsize=9)
ax.set_xlabel("Weight (%)", fontsize=11)
ax.set_title("AHP Factor Weights", fontsize=13, fontweight="bold")
ax.invert_yaxis()
ax.set_xlim(0, max(vals) * 1.15)
ax.axvline(
    100 / len(keys),
    color="navy",
    linestyle="--",
    linewidth=1,
    label=f"Equal weight ({100 / len(keys):.1f}%)",
)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "AHP_Weights.png"), dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
pixel_area_ha = TARGET_RES**2 / 10000
total_ha = total_valid * pixel_area_ha
summary = pd.DataFrame(
    {
        "Class": unique.astype(int),
        "Description": [suit_desc[int(value)] for value in unique],
        "Pixels": counts,
        "Area (ha)": counts * pixel_area_ha,
        "Area (%)": counts / total_valid * 100,
    }
)
print(summary.round(2).to_string(index=False))


In [ ]:
import matplotlib

OUTPUT_MAP_IMG = os.path.join(OUTPUT_DIR, "Solar_Suitability_Classified_Map.tif")
plot_cl = np.where(final_classified == NODATA, np.nan, final_classified)
fig = plt.figure(frameon=False)
fig.set_size_inches(new_width / 100, new_height / 100)
ax = plt.Axes(fig, [0.0, 0.0, 1.0, 1.0])
ax.set_axis_off()
fig.add_axes(ax)
ax.imshow(
    plot_cl, cmap=CLASS_CMAP, norm=CLASS_NORM, interpolation="nearest", aspect="auto"
)
plt.savefig(
    OUTPUT_MAP_IMG,
    dpi=100,
    bbox_inches="tight",
    pad_inches=0,
    transparent=False,
    format="tiff",
)
plt.close()


In [ ]:
SENSITIVITY_RESULTS = []
TEST_VARS = ("DNI", "LULC", "R_Rabi", "Temperature", "Cloudy_Days")
orig_valid = weighted_sum[~nodata_mask]
orig_mean = orig_valid.mean()
orig_std = orig_valid.std()
orig_classes = final_classified[~nodata_mask]
orig_unique, orig_counts = np.unique(orig_classes.astype(int), return_counts=True)
orig_class_pct = {
    int(u): c / len(orig_classes) * 100 for u, c in zip(orig_unique, orig_counts)
}
for var in TEST_VARS:
    degraded = {}
    for key, arr in RECLASSIFIED.items():
        if key == var:
            degraded_arr = arr.copy()
            degraded_arr[arr != NODATA] = 1.0
            degraded[key] = degraded_arr
        else:
            degraded[key] = arr
    deg_weighted = np.zeros(REF_SHAPE, dtype=np.float32)
    for key, arr in degraded.items():
        weight = WEIGHTS.get(key, 0)
        safe_arr = np.where(arr == NODATA, 0.0, arr)
        deg_weighted += safe_arr * weight
    deg_weighted[nodata_mask] = NODATA
    deg_valid = deg_weighted[~nodata_mask]
    deg_mean = deg_valid.mean()
    score_drop = orig_mean - deg_mean
    pct_drop = score_drop / orig_mean * 100
    vmin, vmax = (deg_valid.min(), deg_valid.max())
    if vmax > vmin:
        interval = (vmax - vmin) / 5
        breaks = [vmin + interval * i for i in range(1, 5)]
        deg_cls = np.digitize(deg_valid, breaks, right=False) + 1
        deg_cls = np.clip(deg_cls, 1, 5)
    else:
        deg_cls = np.full_like(deg_valid, 3, dtype=int)
    deg_unique, deg_counts = np.unique(deg_cls.astype(int), return_counts=True)
    deg_class_pct = {
        int(u): c / len(deg_cls) * 100 for u, c in zip(deg_unique, deg_counts)
    }
    SENSITIVITY_RESULTS.append(
        {
            "Variable": var,
            "Weight_%": WEIGHTS.get(var, 0) * 100,
            "Orig_Mean": orig_mean,
            "Deg_Mean": deg_mean,
            "Score_Drop": score_drop,
            "Pct_Drop_%": pct_drop,
            "Deg_Class_%": deg_class_pct,
        }
    )
sensitivity_summary = pd.DataFrame(SENSITIVITY_RESULTS).drop(columns="Deg_Class_%")
print(sensitivity_summary.round(4).to_string(index=False))


In [ ]:
import matplotlib

matplotlib.rcParams["font.family"] = "Times New Roman"
matplotlib.rcParams["font.size"] = 12
OUTPUT_SENSITIVITY_IMG = os.path.join(OUTPUT_DIR, "Sensitivity_Analysis.tif")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
ax = axes[0]
vars_ = [r["Variable"] for r in SENSITIVITY_RESULTS]
drops = [r["Pct_Drop_%"] for r in SENSITIVITY_RESULTS]
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(vars_)))
bars = ax.barh(vars_, drops, color=colors, edgecolor="grey", linewidth=0.5)
ax.bar_label(bars, fmt="%.2f%%", padding=3, fontsize=12, fontfamily="Times New Roman")
ax.set_xlabel("Suitability Score Drop (%)", fontsize=12, fontfamily="Times New Roman")
ax.set_title(
    "Impact of Degrading Each Variable\nto Most Unsuitable (Class 1)",
    fontsize=12,
    fontweight="bold",
    fontfamily="Times New Roman",
)
ax.invert_yaxis()
ax.set_xlim(0, max(drops) * 1.2)
ax.axvline(
    np.mean(drops),
    color="navy",
    linestyle="--",
    linewidth=1,
    label=f"Mean impact ({np.mean(drops):.1f}%)",
)
ax.legend(fontsize=12, prop={"family": "Times New Roman", "size": 12})
ax.tick_params(labelsize=12)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontfamily("Times New Roman")
ax = axes[1]
x = np.arange(5) + 1
width = 0.12
colors2 = plt.cm.tab10(np.linspace(0, 0.5, len(SENSITIVITY_RESULTS)))
for i, r in enumerate(SENSITIVITY_RESULTS):
    deg_pcts = [r["Deg_Class_%"].get(c, 0) for c in range(1, 6)]
    ax.bar(
        x + i * width - width * len(SENSITIVITY_RESULTS) / 2,
        deg_pcts,
        width,
        label=r["Variable"],
        color=colors2[i],
        edgecolor="grey",
        linewidth=0.3,
    )
orig_pcts = [orig_class_pct.get(c, 0) for c in range(1, 6)]
ax.step(
    x,
    orig_pcts,
    where="mid",
    color="black",
    linewidth=2,
    linestyle="--",
    label="Original",
)
ax.set_xlabel("Suitability Class", fontsize=12, fontfamily="Times New Roman")
ax.set_ylabel("% of Valid Pixels", fontsize=12, fontfamily="Times New Roman")
ax.set_xticks(x)
ax.set_xticklabels(
    [
        "1\nMost\nUnsuit.",
        "2\nMod.\nUnsuit.",
        "3\nNeutral",
        "4\nMod.\nSuit.",
        "5\nMost\nSuit.",
    ],
    fontsize=12,
    fontfamily="Times New Roman",
)
ax.set_title(
    "Class Distribution After Degrading\nEach Variable",
    fontsize=12,
    fontweight="bold",
    fontfamily="Times New Roman",
)
ax.legend(
    fontsize=12, loc="upper right", prop={"family": "Times New Roman", "size": 12}
)
ax.tick_params(labelsize=12)
for label in ax.get_yticklabels():
    label.set_fontfamily("Times New Roman")
plt.suptitle(
    "Sensitivity Analysis : Variable Degradation Test",
    fontsize=12,
    fontweight="bold",
    fontfamily="Times New Roman",
)
plt.tight_layout()
plt.savefig(OUTPUT_SENSITIVITY_IMG, dpi=200, bbox_inches="tight", format="tiff")
plt.close()
